# Spenso Python API Documentation

A comprehensive Python tensor library for symbolic and numerical tensor computations, with a focus on physics applications.

## Overview

The Spenso Python API provides powerful tools for:

- **Tensor Algebra**: Dense and sparse tensors with flexible data types
- **Symbolic Computation**: Integration with Symbolica for tensors with symbolic expressions
- **Network Operations**: Tensor networks for optimized computation graphs
- **Physics Applications**: Built-in support for HEP tensors (gamma matrices, color structures, etc.)
- **Performance**: Compiled evaluators for high-speed numerical computation

## Installation

In [ ]:
!pip install symbolica

## Quick Start

In [1]:
from symbolica import S
from symbolica.community.spenso import (
    Tensor,
    TensorIndices,
    TensorStructure,
    TensorName,
    Representation,
    Slot,
    TensorNetwork,
    TensorLibrary,
)

lor = Representation.mink(3)

mu = lor("mu")
nu = lor("nu")
indices = TensorIndices(mu, nu)

data = [1.0, 0.0, 0.0,
        0.0, 1.0, 0.0,
        0.0, 0.0, 1.0]

tensor = Tensor.dense(indices, data)

x, y = S("x", "y")
T = TensorName("T")
symbolic_tensor = T(mu, nu)

symbolic_tensor

PermutedStructure { structure: 
╭──────────────────┬─────────────────┬─────────────┬────────────────────────────────────────────────────╮
│ spenso_python::T │                 │             │                                                    │
├──────────────────┼─────────────────┼─────────────┼────────────────────────────────────────────────────┤
│ 0                │ InlineMetric(0) │ Concrete(3) │ Symbol(SerializableSymbol { symbol: spynso3::mu }) │
│ 1                │ InlineMetric(0) │ Concrete(3) │ Symbol(SerializableSymbol { symbol: spynso3::nu }) │
╰──────────────────┴─────────────────┴─────────────┴────────────────────────────────────────────────────╯, rep_permutation: Permutation { map: [0, 1], inv: [0, 1] }, index_permutation: Permutation { map: [0, 1], inv: [0, 1] } }

## Tensor Class

In [2]:
from symbolica import S
from symbolica.community.spenso import Representation, Tensor, TensorIndices

euc3 = Representation.euc(3)
structure = TensorIndices(euc3(1), euc3(2))

data = [1.0,2.0,3.0,
        4.0,5.0,6.0,
        7.0,8.0,9.0]

tensor = Tensor.dense(structure, data)

sparse_tensor = Tensor.sparse(structure, float)
sparse_tensor[0,0] = 1.0
sparse_tensor[1,1] = 2.0

x,y = S("x","y")

sym_data = [x, y, x*y, x+y]

euc2 = Representation.euc(2)
structure_2x2 = TensorIndices(euc2(1), euc2(2))

sym_tensor = Tensor.dense(structure_2x2, sym_data)

tensor[0] = 10.0
tensor.to_sparse()

sym_tensor

Spensor(
() [0 1]() [0 1][0, 0]: x
[0, 1]: y
[1, 0]: x*y
[1, 1]: x+y
)

## Representations

In [4]:
from symbolica.community.spenso import Representation

euclidean = Representation.euc(4)
minkowski = Representation.mink(4)
color_fund = Representation.cof(3)
color_adj = Representation.coad(8)
bispinor = Representation.bis(4)

custom = Representation("MyRep",5,is_self_dual=False)

mu_slot = euclidean("mu")

metric = euclidean.g("mu","nu")
flat_metric = euclidean.flat("mu","nu")
identity = custom.id("mu","nu")

print(metric)

g(euc(4,mu),euc(4,nu))


## TensorName

In [6]:
from symbolica.community.spenso import TensorName, Representation

# Create tensor names
T = TensorName("T")  # Basic tensor
g = TensorName("g", is_symmetric=True)  # Symmetric tensor
F = TensorName("F", is_antisymmetric=True)  # Antisymmetric tensor

# Predefined physics tensors
gamma = TensorName.gamma  # Dirac gamma matrices
sigma = TensorName.sigma  # Pauli matrices
f_abc = TensorName.f  # SU(N) structure constants

# Use with indices
rep = Representation.mink(3)
mu = rep("mu")
nu = rep("nu")
tensor_expr = T(mu, nu)  # Creates TensorIndices T(μ,ν)

print(tensor_expr)


T(mink(3,mu),mink(3,nu))


In [8]:
from symbolica import S
from symbolica.community.spenso import (
    TensorIndices,
    TensorStructure,
    TensorName,
    Representation,
)

# TensorIndices: For tensors with specific abstract indices
rep = Representation.mink(3)
mu = rep("mu")
nu = rep("nu")

# TensorStructure: For tensor templates without specific indices
structure = TensorStructure(rep, rep)

# Named structures
T = TensorName("T")
named_structure = TensorStructure(rep, rep, name=T)

# Convert structure to indices
concrete_indices = structure.index("a", "b")  # Assign indices a, b

# Create symbolic expressions
symbolic_expr = named_structure.symbolic("mu", "nu")  # T(μ,ν)

# With additional arguments
x = S("x")
expr_with_args = named_structure.symbolic(x, ";", "mu", "nu")  # T(x; μ,ν)

print(symbolic_expr)
print(expr_with_args)
print(concrete_indices)

T(mink(3,mu),mink(3,nu))
T(x,mink(3,mu),mink(3,nu))
(mink(3,a),mink(3,b))


## TensorNetwork

In [11]:
from symbolica import E
from symbolica.community.spenso import (
    TensorNetwork,
    TensorName,
    ExecutionMode,
    Representation,
)

# TensorIndices: For tensors with specific abstract indices
rep = Representation.mink(3)
mu = rep("mu")
nu = rep("nu")
x = E("sin(x)")
T = TensorName("T")
expr = x * T(mu, nu)
network = TensorNetwork(expr)
# Controlled execution
network.execute(mode=ExecutionMode.Scalar)  # Only scalar operations
result = network.result_tensor()
print(result)

[0, 0]: sin(x)*T(cind(0,0))
[0, 1]: sin(x)*T(cind(0,1))
[0, 2]: sin(x)*T(cind(0,2))
[1, 0]: sin(x)*T(cind(1,0))
[1, 1]: sin(x)*T(cind(1,1))
[1, 2]: sin(x)*T(cind(1,2))
[2, 0]: sin(x)*T(cind(2,0))
[2, 1]: sin(x)*T(cind(2,1))
[2, 2]: sin(x)*T(cind(2,2))



## Tensor Library

In [14]:
from symbolica import S
from symbolica.community.spenso import (
    TensorLibrary,
    LibraryTensor,
    TensorName as N,
    TensorStructure,
    Representation,
)

# Create library
lib = TensorLibrary()

# Register tensors
rep = Representation.euc(2)

sigma_x = S("σ_x")
structure = TensorStructure(rep, rep, name=sigma_x)
pauli_x = LibraryTensor.dense(structure, [0.0, 1.0, 1.0, 0.0])
lib.register(pauli_x)

# Access registered tensors
sigma_structure = lib[sigma_x]

# HEP library with standard physics tensors
hep_lib = TensorLibrary.hep_lib()
gamma_structure = hep_lib[N.gamma()]


print(gamma_structure)

gamma[bis(4),bis(4),mink(4)]


## Evaluation and performance

### Symbolic Evaluation

In [15]:
# Create symbolic tensor
x, y = S('x','y')
structure = TensorIndices(Representation.euc(2)(1), Representation.euc(2)(1))
tensor = Tensor.dense(structure, [x*y, x+y])

# Create evaluator
evaluator = tensor.evaluator(
    constants={},           # Constant values
    funs={},               # Function definitions
    params=[x, y],         # Variable parameters
    iterations=100,        # Optimization iterations
    n_cores=4             # Parallel cores
)

# Evaluate for multiple parameter sets
inputs = [[1.0, 2.0], [3.0, 4.0]]  # x,y values
results = evaluator.evaluate(inputs)

print(results)  


OverflowError: Data length does not match shape

In [16]:
# Compile for maximum performance
compiled_evaluator = evaluator.compile(
    function_name="fast_eval",
    filename="tensor_eval.cpp",
    library_name="tensor_lib",
    optimization_level=3
)

# Use compiled evaluator for complex inputs
complex_inputs = [[1.0+2.0j, 3.0+0.0j]]
results = compiled_evaluator.evaluate_complex(complex_inputs)

NameError: name 'evaluator' is not defined

## Physics

In [19]:
from symbolica.community.spenso import TensorLibrary, TensorName, Representation

# Load HEP library
hep_lib = TensorLibrary.hep_lib()

# Standard HEP tensors
gamma = TensorName.gamma()  # Dirac gamma matrices
gamma5 = TensorName.gamma5  # γ₅ matrix
sigma = TensorName.sigma  # Pauli matrices
f_abc = TensorName.f  # SU(N) structure constants
t_a = TensorName.t  # SU(N) generators

# Representations
lorentz = Representation.mink(4)  # Lorentz indices
spinor = Representation.bis(4)  # Dirac spinor

# Build physics expressions
mu = lorentz("mu")
nu = lorentz("nu")
alpha = spinor("alpha")
beta = spinor("beta")

# Gamma matrix trace: Tr(γᵘγᵥ)
gamma_trace = gamma(alpha, beta, mu) * gamma(beta, alpha, nu)

print(gamma_trace)

gamma(bis(4,alpha),bis(4,beta),mink(4,mu))*gamma(bis(4,beta),bis(4,alpha),mink(4,nu))


Tensor contraction

In [25]:
from symbolica.community.spenso import (
    TensorLibrary,
    TensorName,
    Representation,
    TensorNetwork,
)

rep = Representation.mink(4)
mu = rep("mu")
nu = rep("nu")
rho = rep("rho")

# Metric contraction: gᵘᵛ gᵤᵥ = 4
g = TensorName.g()
contraction = g(mu, nu) * g(mu, nu)  # Repeated indices contract

# Network evaluation
network = TensorNetwork(contraction)
network.execute()
result = network.result_scalar()  # Should give 4 for 4D

print(result)

4.00000000000000
